# Robo de vehículos en México y Jalisco
## Validación de las tres hipótesis del proyecto final

**César Roberto Murillo Figueroa — ID 0283143**
Especialidad en Ciencia de Datos · Universidad Panamericana, campus Guadalajara
Visualización de Datos y Narración de Historias · Dr. Rodolfo Rubén Álvarez González

---

### Hipótesis planteadas al inicio del proyecto

| # | Hipótesis | Veredicto |
|---|---|---|
| **H1 · Patrón horario** | La mayoría de los robos ocurren durante la noche o madrugada, aprovechando la menor visibilidad y vigilancia vial. | **Se refuta** |
| **H2 · Modus operandi** | Predomina el robo a mano armada (con violencia) por encima del robo de autos estacionados en vía pública. | **Se refuta** |
| **H3 · Factor socioeconómico** | Ocurren con mayor frecuencia en zonas de altos ingresos debido a la presencia de vehículos de mayor valor comercial. | **Se confirma** |

### Qué hace este notebook

Reconstruye de principio a fin la evidencia de las tres hipótesis: descarga la fuente
socioeconómica desde el portal de CONAPO, cruza espacialmente los robos georreferenciados
con los polígonos de colonia, y genera las cinco figuras del reporte. Corre completo sin
intervención salvo por la carga inicial de dos archivos (celda 2).

### Fuentes de datos

| Fuente | Institución | Qué aporta | Periodo |
|---|---|---|---|
| Base georreferenciada de delitos | IIEG Jalisco | Hora, fecha, colonia y coordenadas de cada robo | jul-2025 a jul-2026 |
| Incidencia delictiva estatal | SESNSP | Modalidad con/sin violencia | 2015–2025 |
| Índice de marginación urbana por colonia | CONAPO | Marginación y población por colonia | Censo 2020 |

---
## 1. Preparación del entorno

`geopandas` no viene preinstalado en Colab; las demás sí. La instalación tarda cerca de un minuto.

In [ ]:
# Instala solo lo que falta en Colab. Si ya está, pip no hace nada.
!pip install -q geopandas pyogrio 2>/dev/null

import warnings; warnings.filterwarnings('ignore')
import os, zipfile, urllib.request, pathlib
import pandas as pd, numpy as np, geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

print("pandas", pd.__version__, "| geopandas", gpd.__version__)

---
## 2. Carga de los dos archivos del proyecto

Se necesitan dos archivos que ya forman parte del proyecto:

1. **`Base_Robo_Vehiculos_Jalisco.xlsx`** — la base limpia de 11 696 robos georreferenciados,
   producto de la actividad de anonimización (filtrado de los 52 366 registros originales del
   IIEG a los cuatro delitos de robo vehicular, con la variable `hora` ya normalizada a HH:MM).
2. **`Estatal-Delitos-2015-2025_jul2026.csv`** — incidencia delictiva estatal del SESNSP.

El notebook los busca primero en Google Drive; si no los encuentra, abre el diálogo de carga manual.

In [ ]:
ARCHIVOS = ["Base_Robo_Vehiculos_Jalisco.xlsx", "Estatal-Delitos-2015-2025_jul2026.csv"]

# Carpetas donde buscar, en orden. Agrega aquí tu ruta de Drive si la tienes en otro lado.
CANDIDATAS = [
    # rutas del repositorio de GitHub, se ejecute desde la raíz o desde notebooks/
    "data/processed", "data/raw", "../data/processed", "../data/raw",
    ".", "..", "/content",
    "/content/drive/MyDrive/UP/Proyecto_Robo_Vehiculos",
    "/content/drive/MyDrive/Colab Notebooks",
    "/content/drive/MyDrive",
]

def montar_drive():
    try:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        return True
    except Exception:
        return False

def localizar(nombre):
    # Devuelve la ruta del archivo; lo pide por carga manual si no aparece.
    for base in CANDIDATAS:
        p = os.path.join(base, nombre)
        if os.path.exists(p):
            return p
    # búsqueda recursiva en Drive como último recurso automático
    for raiz in ["/content/drive/MyDrive"]:
        if os.path.isdir(raiz):
            for dirpath, _, files in os.walk(raiz):
                if nombre in files:
                    return os.path.join(dirpath, nombre)
    try:
        from google.colab import files
        print(f"No encontré «{nombre}». Selecciónalo para subirlo:")
        subidos = files.upload()
        if nombre in subidos:
            return nombre
        if subidos:
            return list(subidos)[0]
    except Exception:
        pass
    raise FileNotFoundError(
        f"Falta «{nombre}». Súbelo a Colab o colócalo en una de estas carpetas: {CANDIDATAS}")

montar_drive()
RUTA_ROBOS  = localizar(ARCHIVOS[0])
RUTA_SESNSP = localizar(ARCHIVOS[1])
SALIDA = pathlib.Path("figuras"); SALIDA.mkdir(exist_ok=True)
print("Robos :", RUTA_ROBOS)
print("SESNSP:", RUTA_SESNSP)

In [ ]:
robos = pd.read_excel(RUTA_ROBOS)
robos['fecha'] = pd.to_datetime(robos['fecha'])
robos['h']   = pd.to_datetime(robos['hora'].astype(str), format='mixed').dt.hour
robos['dow'] = robos['fecha'].dt.day_of_week   # 0 = lunes

print("Registros:", len(robos))
print("Periodo  :", robos['fecha'].min().date(), "a", robos['fecha'].max().date())
print("\nPor delito:");  print(robos['delito'].value_counts().to_string())
print("\nPor zona:");    print(robos['zona_geografica'].value_counts().to_string())
print("\nValores nulos:", robos.isna().sum().sum())

---
## 3. Descarga del Índice de Marginación Urbana por colonia (CONAPO 2020)

Esta es la variable socioeconómica que faltaba para poder evaluar H3. Se descarga directo del
portal de CONAPO, de modo que el notebook es reproducible sin archivos adicionales.

Se usan dos insumos: la **base de datos por colonia** (`IMC_2020.zip`) y la **cartografía digital**
(`imc2020_shp.zip`), que trae los polígonos necesarios para el cruce espacial.

> **Se eligió el nivel colonia y no el nivel AGEB** porque el archivo de AGEB publica el índice
> pero no su geometría, de modo que cruzarlo con las coordenadas de los robos exigiría además
> descargar el Marco Geoestadístico del INEGI. El archivo de colonia sí incluye los polígonos y
> coincide con la granularidad de la base del IIEG, que ya registra la colonia de cada hecho.

In [ ]:
URLS = {
 "IMC_2020.zip":   "https://conapo.segob.gob.mx/work/models/CONAPO/Datos_Abiertos/Colonia/IMC_2020.zip",
 "imc2020_shp.zip":"https://conapo.segob.gob.mx/work/models/CONAPO/Datos_Abiertos/Colonia/imc2020_shp.zip",
}
CONAPO = pathlib.Path("conapo"); CONAPO.mkdir(exist_ok=True)

for nombre, url in URLS.items():
    destino = CONAPO / nombre
    if not destino.exists():
        print("Descargando", nombre, "...")
        urllib.request.urlretrieve(url, destino)
    carpeta = CONAPO / destino.stem
    if not carpeta.exists():
        with zipfile.ZipFile(destino) as z:
            z.extractall(carpeta)
    print(f"  {nombre}: {destino.stat().st_size/1e6:.1f} MB")

RUTA_SHP = next((CONAPO / "imc2020_shp").rglob("*.shp"))
print("\nShapefile:", RUTA_SHP)

### Advertencia sobre la escala del índice

El campo `IM_2020` de CONAPO 2020 está **invertido respecto a lo que sugiere su nombre**: un valor
alto significa marginación **baja**. La verificación está en la celda siguiente — los indicadores
crudos (hacinamiento, población sin educación básica) confirman que el *grado* `GM_2020` sí corre
en el sentido habitual, pero el *índice numérico* no. Quien use `IM_2020` directamente sin revisar
esto obtiene la conclusión exactamente al revés.

In [ ]:
colonias = gpd.read_file(RUTA_SHP)
colonias['CVE_ENT'] = colonias['CVE_ENT'].astype(str).str.zfill(2)
jal = colonias[colonias['CVE_ENT'] == '14'].to_crs("EPSG:4326")

ORDEN = ['Muy bajo', 'Bajo', 'Medio', 'Alto', 'Muy alto']
print("Comprobación de la dirección del índice:\n")
chk = jal.groupby('GM_2020')[['IM_2020', 'OVHAC', 'SBASC']].mean().reindex(ORDEN).round(2)
chk.columns = ['IM_2020 (índice)', '% hacinamiento', '% sin educación básica']
print(chk.to_string())
print("\nEl hacinamiento SUBE de «Muy bajo» a «Muy alto» (grado correcto),")
print("pero el índice IM_2020 BAJA: por eso se trabaja con GM_2020, no con IM_2020.")

---
## 4. Cruce espacial: cada robo con la colonia donde ocurrió

Se usa un *point-in-polygon* sobre las coordenadas y no una unión por nombre de colonia, que sería
frágil: los nombres del IIEG y de CONAPO difieren en acentos, abreviaturas y variantes
(«Zona Centro» vs «Centro»), y hay nombres repetidos en municipios distintos.

In [ ]:
puntos = gpd.GeoDataFrame(
    robos, geometry=gpd.points_from_xy(robos['x'], robos['y']), crs="EPSG:4326")

CAMPOS = ['CVE_COL','COLONIA','NOM_MUN','POBTOT','IM_2020','GM_2020','OVHAC','SBASC']
cruce = gpd.sjoin(puntos, jal[CAMPOS + ['geometry']], how='left', predicate='within')
cruce = cruce[~cruce.index.duplicated(keep='first')]      # un punto en 2 polígonos: se queda el 1.º

enlazados = cruce['GM_2020'].notna()
print(f"Robos enlazados a una colonia: {enlazados.sum():,} de {len(cruce):,} "
      f"({enlazados.mean()*100:.1f} %)")
print("\nSin enlace, por zona:")
print(cruce.loc[~enlazados, 'zona_geografica'].value_counts().to_string())

# Base enriquecida, por si se quiere exportar al reporte
base_enriquecida = cruce.drop(columns=['geometry', 'index_right']).rename(columns={
    'COLONIA':'colonia_conapo', 'NOM_MUN':'municipio_conapo', 'POBTOT':'pob_colonia_2020',
    'IM_2020':'indice_marg_2020', 'GM_2020':'grado_marg_2020',
    'OVHAC':'pct_hacinamiento', 'SBASC':'pct_sin_basica'})
base_enriquecida.to_excel("Base_Robo_Vehiculos_Jalisco_con_Marginacion.xlsx", index=False)
print("\nGuardado: Base_Robo_Vehiculos_Jalisco_con_Marginacion.xlsx")

In [ ]:
# Tabla colonia x robos, que alimenta las dos figuras de H3
AMG = ['Guadalajara','Zapopan','San Pedro Tlaquepaque','Tonalá','Tlajomulco de Zúñiga',
       'El Salto','Juanacatlán','Ixtlahuacán de los Membrillos','Zapotlanejo']

por_colonia = (jal.drop_duplicates('CVE_COL')[['CVE_COL','COLONIA','NOM_MUN','GM_2020','IM_2020','POBTOT','geometry']]
                  .merge(cruce.dropna(subset=['CVE_COL']).groupby('CVE_COL').size().rename('robos'),
                         on='CVE_COL', how='left'))
por_colonia['robos'] = por_colonia['robos'].fillna(0)
amg = por_colonia[por_colonia['NOM_MUN'].isin(AMG)].copy()

tabla_h3 = amg.groupby('GM_2020').agg(colonias=('CVE_COL','size'),
                                      poblacion=('POBTOT','sum'),
                                      robos=('robos','sum')).reindex(ORDEN)
tabla_h3['tasa_100k'] = (tabla_h3['robos'] / tabla_h3['poblacion'] * 1e5).round(1)
tabla_h3['%_robos']   = (tabla_h3['robos'] / tabla_h3['robos'].sum() * 100).round(1)
print("Robos por grado de marginación — Área Metropolitana de Guadalajara\n")
print(tabla_h3.to_string())

---
## 5. Estilo gráfico

Un solo lugar define colores y adornos, para que las cinco figuras se lean como un mismo sistema.

Los criterios aplicados siguen a Knaflic (*Storytelling with Data*): **título-conclusión** en cada
figura (el título afirma el hallazgo, no describe los ejes), color con intención y sin *chartjunk*.
La paleta se verificó con un validador de accesibilidad: separación bajo simulación de daltonismo
(protanopía y deuteranopía) por encima del umbral ΔE ≥ 8 en OKLab, y contraste suficiente contra el
fondo. Para variables ordenadas (grado de marginación, intensidad en el mapa) se usa una rampa de
**un solo tono** de claro a oscuro, nunca un arcoíris.

In [ ]:
SURF='#fcfcfb'; INK='#0b0b0b'; INK2='#52514e'; MUTED='#898781'; GRID='#e1e0d9'; BASE='#c3c2b7'
BLUE='#2a78d6'; ORANGE='#eb6834'; GRISFONDO='#e4e3dd'
RAMPA_ORD = ['#86b6ef','#5598e7','#2a78d6','#1c5cab','#0d366b']   # ordinal: claro -> oscuro
RAMPA_SEQ = ['#cde2fb','#86b6ef','#3987e5','#1c5cab','#0d366b']   # secuencial (mapa y heatmap)

plt.rcParams.update({
 'figure.facecolor':SURF,'axes.facecolor':SURF,'savefig.facecolor':SURF,
 'font.family':'DejaVu Sans','text.color':INK,'axes.labelcolor':INK2,
 'xtick.color':MUTED,'ytick.color':MUTED,'axes.edgecolor':BASE,'axes.linewidth':.8,
 'xtick.labelsize':9,'ytick.labelsize':9,'grid.color':GRID,'grid.linewidth':.8,
 'grid.linestyle':'-','figure.dpi':110,'savefig.dpi':200,'savefig.bbox':'tight',
 'legend.frameon':False,'legend.fontsize':9.5})

def marco(ax, ygrid=True):
    ax.set_axisbelow(True); ax.yaxis.grid(ygrid); ax.xaxis.grid(False)
    for s in ('top','right'): ax.spines[s].set_visible(False)

def encabeza(ax, titulo, subtitulo, ty=1.14, sy=1.045, fs=14):
    ax.text(0, ty, titulo,    transform=ax.transAxes, fontsize=fs, fontweight='bold', color=INK, va='bottom')
    ax.text(0, sy, subtitulo, transform=ax.transAxes, fontsize=9.5, color=INK2, va='bottom')

def nota(fig, texto):
    fig.text(0, -0.055, texto, fontsize=9, color=INK2, style='italic', va='top')

FTE = "Fuente: IIEG Jalisco · base georreferenciada jul-2025 a jul-2026 · n = 11 696"
print("Estilo cargado.")

---
# H1 · Patrón horario

> *«La mayoría de los robos ocurren durante la noche o madrugada aprovechando la menor
> visibilidad y vigilancia vial.»*

### Figura 1 — Robos por hora del día

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
conteo = robos['h'].value_counts().reindex(range(24), fill_value=0)
pct = conteo / conteo.sum() * 100

ax.axvspan(-0.5, 5.5, color=GRISFONDO, zorder=0, lw=0)
ax.bar(conteo.index, conteo.values, color=BLUE, width=.72, zorder=3)
ax.text(2.5, conteo.max()*1.02,
        f"Madrugada 00–05\nsólo {pct.loc[0:5].sum():.1f} % de los robos",
        ha='center', va='top', fontsize=9.5, color=INK2, linespacing=1.5)
for hh, etq in ((7, "entrada al trabajo"), (21, "regreso a casa")):
    ax.annotate(f"{conteo[hh]:,}".replace(',', '\u202f'), (hh, conteo[hh]),
                textcoords='offset points', xytext=(0, 5), ha='center',
                fontsize=9.5, fontweight='bold', color=INK)
    ax.annotate(etq, (hh, conteo[hh]), textcoords='offset points', xytext=(0, 22),
                ha='center', fontsize=9, color=INK2)

marco(ax)
ax.set_xticks(range(24)); ax.set_xticklabels([f"{i:02d}" for i in range(24)])
ax.set_xlim(-0.8, 23.8); ax.set_ylim(0, conteo.max()*1.20)
ax.set_xlabel("Hora del día", fontsize=10); ax.set_ylabel("Robos", fontsize=10)
encabeza(ax, "La madrugada es cuando MENOS roban vehículos",
         "Robos de y a vehículos por hora del día · " + FTE)
nota(fig, "H1 se refuta: el patrón es bimodal y sigue los horarios de movilidad urbana, no la oscuridad.")
fig.savefig(SALIDA/"fig1_h1_hora.png"); plt.show()

print(f"Diurno 07:00–18:59 : {pct.loc[7:18].sum():.1f} %")
print(f"Nocturno 19:00–06:59: {100 - pct.loc[7:18].sum():.1f} %")
print(f"Madrugada 00:00–05:59: {pct.loc[0:5].sum():.1f} %")

**Lectura.** El reparto día/noche queda en un empate casi perfecto (49.5 % diurno), así que la
hipótesis no se sostiene ni siquiera en su versión más laxa. Pero el dato relevante es otro: la
distribución es **bimodal**, con un pico a las 07:00 y otro mayor entre las 20:00 y las 22:00, y un
mínimo claro entre las 02:00 y las 04:00. El robo sigue el ritmo de la movilidad urbana —cuando hay
vehículos circulando y estacionándose— y no la oscuridad.

Conviene anotar un detalle de calidad de datos: el 64 % de los registros cae en minuto `:00` o
`:30`, señal de que la hora se captura redondeada. Eso no mueve el análisis por hora, pero impide
afirmar nada a escala de minutos. Las horas `00:00` exactas son sólo 51 registros, de modo que no
hay un valor por defecto de captura inflando la medianoche.

### Figura 2 — Hora contra día de la semana

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
DIAS = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']
M = (robos.pivot_table(index='dow', columns='h', values='id_registro', aggfunc='count')
          .reindex(index=range(7), columns=range(24)).fillna(0))

cortes = [0, *np.quantile(M.values, [.25, .5, .75, .92]), M.values.max()]
im = ax.imshow(M.values, cmap=ListedColormap(RAMPA_SEQ),
               norm=BoundaryNorm(cortes, len(RAMPA_SEQ)), aspect='auto')

ax.set_xticks(range(24)); ax.set_xticklabels([f"{i:02d}" for i in range(24)])
ax.set_yticks(range(7));  ax.set_yticklabels(DIAS, fontsize=9.5)
ax.set_xticks(np.arange(-.5, 24, 1), minor=True); ax.set_yticks(np.arange(-.5, 7, 1), minor=True)
ax.grid(which='minor', color=SURF, linewidth=2); ax.tick_params(which='minor', length=0); ax.grid(False)
for s in ax.spines.values(): s.set_visible(False)

cb = fig.colorbar(im, ax=ax, pad=.012, fraction=.022, ticks=[round(c) for c in cortes])
cb.outline.set_visible(False); cb.ax.tick_params(labelsize=8, length=0)
cb.set_label("Robos por celda", fontsize=9, color=INK2)
ax.set_xlabel("Hora del día", fontsize=10)

finsem = M.loc[5:6].values.sum() / M.values.sum() * 100
encabeza(ax, f"Sábado y domingo concentran sólo {finsem:.0f} % de los robos de la semana",
         "Robos por hora y día de la semana · " + FTE, ty=1.19, sy=1.06)
nota(fig, "La forma bimodal se repite los siete días. Nota: si «fecha» fuera la de la denuncia y no la del hecho,\n"
          "la caída de sábado y domingo sería un artefacto administrativo — conviene verificarlo con el IIEG.")
fig.savefig(SALIDA/"fig2_h1_heatmap.png"); plt.show()

forma = (M.div(M.sum(axis=1), axis=0) * 100)
print("Peso del pico matutino (06–09 h) dentro de cada día, en %:")
print(forma.loc[:, 6:9].sum(axis=1).round(1).rename(index=dict(enumerate(DIAS))).to_string())

**Lectura.** Lo que cambia el fin de semana es el **volumen**, no el horario: el pico de las 21:00
está presente los siete días, y el matutino también (entre 19.9 % y 26.2 % del total de cada día,
sin que el fin de semana quede fuera de ese rango). La forma del día es notablemente estable.

**Reserva metodológica.** La caída de sábado y domingo coincide con lo que reportó el IIEG en su
estudio de 2019 mediante un modelo GLM quasipoisson, pero sigue abierta una duda: si el campo
`fecha` corresponde a la **denuncia** y no al **hecho**, la caída sería un artefacto administrativo
—las agencias del ministerio público reciben menos denuncias en fin de semana— y no un patrón
delictivo. Mientras no se confirme con el IIEG, esta figura se interpreta con cautela.

---
# H2 · Modus operandi

> *«Predomina el robo a mano armada (con violencia) por encima del robo de autos estacionados
> en vía pública.»*

Aquí se cambia de fuente: la base del IIEG no distingue si hubo violencia, así que se recurre al
SESNSP, que clasifica cada carpeta de investigación por modalidad. Se suman los doce meses de cada
año y se separa el subtipo «Robo de vehículo automotor».

### Figura 3 — Composición del robo, 2015–2025

In [ ]:
sesnsp = pd.read_csv(RUTA_SESNSP, encoding='latin-1')
sesnsp.columns = [c.replace('A\xf1o', 'anio') for c in sesnsp.columns]
MESES = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
         'Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']
sesnsp['total'] = sesnsp[MESES].sum(axis=1)

veh = sesnsp[sesnsp['Subtipo de delito'].str.contains('veh.culo automotor', case=False,
                                                      na=False, regex=True)].copy()
veh['viol'] = np.where(veh['Modalidad'].str.contains('Con violencia'),
                       'Con violencia', 'Sin violencia')

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), sharey=True)
paneles = [("Jalisco", veh[veh['Entidad'].str.contains('Jalisco', na=False)]), ("Nacional", veh)]

for ax, (nombre, datos) in zip(axes, paneles):
    p  = datos.pivot_table(index='anio', columns='viol', values='total', aggfunc='sum')
    sh = p.div(p.sum(axis=1), axis=0) * 100
    anios = [str(a) for a in sh.index]
    ax.bar(anios, sh['Con violencia'], color=ORANGE, width=.68, zorder=3)
    ax.bar(anios, sh['Sin violencia'] - 0.9, bottom=sh['Con violencia'] + 0.9,
           color=BLUE, width=.68, zorder=3)          # el hueco de 0.9 separa los segmentos
    ax.axhline(50, color=INK, lw=1.2, zorder=5)
    ax.text(-0.45, 52, "50 %", fontsize=9, color=INK, ha='left', va='bottom', fontweight='bold')
    for i, val in enumerate(sh['Con violencia'].values):
        ax.text(i, val/2, f"{val:.0f}", ha='center', va='center',
                fontsize=8.5, color='white', fontweight='bold')
    marco(ax, ygrid=False); ax.set_ylim(0, 100); ax.set_yticks(range(0, 101, 25))
    ax.set_yticklabels([f"{t} %" for t in range(0, 101, 25)])
    ax.tick_params(axis='x', labelsize=8.5)
    ax.set_title(nombre, loc='left', fontsize=11, fontweight='bold', color=INK, pad=8)
    ax.spines['left'].set_visible(False)

axes[0].legend(handles=[Patch(facecolor=ORANGE, label='Con violencia'),
                        Patch(facecolor=BLUE,   label='Sin violencia')],
               loc='lower left', bbox_to_anchor=(0, 1.11), ncol=2)
encabeza(axes[0], "El robo CON violencia nunca ha sido mayoría",
         "Composición del robo de vehículo automotor, 2015–2025 · Fuente: SESNSP, carpetas de investigación",
         ty=1.34, sy=1.24)
nota(fig, "H2 se refuta: el robo sin violencia domina los once años. Pero la proporción violenta en Jalisco pasó de 17.1 % (2016)\n"
          "a 42.9 % (2020) y cerró en 31.8 % (2025): la hipótesis falla en nivel, acierta en tendencia.")
fig.savefig(SALIDA/"fig3_h2_violencia.png"); plt.show()

jal_v = veh[veh['Entidad'].str.contains('Jalisco', na=False)]
t = jal_v.pivot_table(index='anio', columns='viol', values='total', aggfunc='sum')
t['Total'] = t.sum(axis=1)
t['% con violencia'] = (t['Con violencia'] / t['Total'] * 100).round(1)
print("Jalisco — robo de vehículo automotor\n"); print(t.to_string())

**Lectura.** La hipótesis se refuta sin ambigüedad: en los once años de la serie, en Jalisco y a
nivel nacional, el robo **sin** violencia es mayoría. La barra naranja nunca cruza la línea del
50 %. El máximo histórico de Jalisco fue 42.9 % en 2020.

Dicho eso, la hipótesis captó algo real en la dirección del cambio: la proporción violenta en
Jalisco se multiplicó por 2.5 entre 2016 (17.1 %) y 2020 (42.9 %), antes de retroceder a 31.8 % en
2025. **Falla en el nivel, acierta en la tendencia.**

**Sobre el «55 % con violencia» de la presentación inicial.** Esa cifra no se sostiene con datos del
SESNSP y no debe usarse. Lo más probable es que proviniera de la ENVIPE, que mide *víctimas
encuestadas* y no *carpetas de investigación*: son universos distintos y no comparables, porque la
ENVIPE capta también los delitos no denunciados, cuya composición difiere.

---
# H3 · Factor socioeconómico

> *«Ocurren con mayor frecuencia en zonas de altos ingresos debido a la presencia de vehículos
> de mayor valor comercial.»*

Ésta era la hipótesis sin datos: la base del IIEG trae colonia y coordenadas, pero ninguna variable
de ingreso. El índice de CONAPO descargado en la sección 3 la hace evaluable. La medida correcta no
es el conteo de robos —las colonias de ingreso alto no son ni las más numerosas ni las más
pobladas— sino la **tasa por 100 mil habitantes**.

### Figura 4 — Tasa de robo por grado de marginación

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5))
t4 = tabla_h3.copy()

ax.bar(range(5), t4['tasa_100k'].values, color=RAMPA_ORD[::-1], width=.66, zorder=3)
for i, (tasa, rb) in enumerate(zip(t4['tasa_100k'].values, t4['robos'].values)):
    ax.text(i, tasa + 6,  f"{tasa:.0f}", ha='center', fontsize=11, fontweight='bold', color=INK)
    ax.text(i, tasa + 22, f"{int(rb):,} robos".replace(',', '\u202f'),
            ha='center', fontsize=8.5, color=MUTED)

marco(ax); ax.set_xticks(range(5))
ax.set_xticklabels(["Muy bajo\n(más ingreso)", "Bajo", "Medio", "Alto", "Muy alto\n(menos ingreso)"],
                   fontsize=9.5)
ax.set_ylim(0, t4['tasa_100k'].max()*1.26)
ax.set_xlabel("Grado de marginación de la colonia (CONAPO 2020)", fontsize=10, labelpad=8)
ax.set_ylabel("Robos por 100 mil habitantes", fontsize=10)
encabeza(ax, "Donde hay menos marginación se roban el doble de vehículos",
         "Tasa de robo por grado de marginación de la colonia · Área Metropolitana de Guadalajara · jul-2025 a jul-2026\n"
         "Fuentes: IIEG Jalisco (robos) y CONAPO, Índice de marginación urbana por colonia 2020",
         ty=1.19, sy=1.045)
nota(fig, "H3 se confirma: 233 contra 124 robos por 100 mil habitantes entre los extremos. Tres advertencias: el máximo real está en «Bajo»\n"
          "(252), no en «Muy bajo»; el denominador es población y no vehículos; y «Muy alto» reúne sólo 40 colonias y 37 mil habitantes.")
fig.savefig(SALIDA/"fig4_h3_tasa.png"); plt.show()

# Control de robustez: quintiles del índice continuo, calculados DENTRO del AMG.
# Recordatorio: IM_2020 está invertido, así que el quintil más bajo es el MÁS marginado.
sub = amg[amg['POBTOT'] >= 500].copy()
sub['q'] = pd.qcut(sub['IM_2020'], 5,
                   labels=['Q1 más marginado','Q2','Q3','Q4','Q5 menos marginado'])
q = (sub.groupby('q', observed=True)
        .apply(lambda d: pd.Series({'colonias': len(d),
                                    'poblacion': d['POBTOT'].sum(),
                                    'robos': d['robos'].sum(),
                                    'tasa_100k': d['robos'].sum()/d['POBTOT'].sum()*1e5})))
q['tasa_100k'] = q['tasa_100k'].round(1)
print("Tasa por quintil del índice, calculado dentro del AMG (colonias de 500+ habitantes):\n")
print(q.to_string())
print(f"\nRazón Q5 (menos marginado) / Q1 (más marginado): "
      f"{q['tasa_100k'].iloc[-1]/q['tasa_100k'].iloc[0]:.2f}x")

**Lectura.** El gradiente es claro y va en la dirección que anticipaba la hipótesis: **233 robos por
cada 100 mil habitantes** en colonias de marginación muy baja contra **120–124** en las de
marginación alta y muy alta, con los niveles intermedios ordenados entre ambos extremos. Casi el
doble. **H3 se confirma.**

Tres reservas que conviene declarar antes de darla por buena:

1. **El máximo no está en el extremo.** El valor más alto corresponde al grado «Bajo» (252), no a
   «Muy bajo» (233). El gradiente existe, pero no es perfectamente monótono.
2. **El denominador es población, no vehículos.** Las zonas de mayor ingreso tienen más autos por
   habitante, así que parte de la diferencia se explica simplemente porque hay más qué robar. Eso
   no refuta la hipótesis, pero **sí cambia el mecanismo**: la causa inmediata sería la densidad
   vehicular, no «el mayor valor comercial» que proponía el enunciado original. Para separar ambos
   efectos haría falta el parque vehicular por colonia, que no está disponible a esa escala.
3. **Sesgo de denuncia.** En zonas de ingreso alto hay más vehículos asegurados, y la aseguradora
   exige carpeta de investigación para pagar. Eso infla su tasa registrada por una razón
   administrativa, no delictiva.

### Figura 5 — Mapa coroplético del Área Metropolitana de Guadalajara

In [ ]:
mapa = amg[amg['POBTOT'] >= 1000].copy()
mapa['tasa'] = mapa['robos'] / mapa['POBTOT'] * 1e5

CORTES = [0, 50, 150, 300, 600, max(mapa['tasa'].max(), 601)]
ETIQ   = ["0 – 50", "50 – 150", "150 – 300", "300 – 600", "600 o más"]
mapa['clase'] = pd.cut(mapa['tasa'], bins=CORTES, labels=ETIQ, include_lowest=True)
mapa = gpd.GeoDataFrame(mapa, geometry='geometry', crs=jal.crs)

fig, ax = plt.subplots(figsize=(9.5, 7.6))
for etq, color in zip(ETIQ, RAMPA_SEQ):
    trozo = mapa[mapa['clase'] == etq]
    if len(trozo):
        trozo.plot(ax=ax, color=color, edgecolor=SURF, linewidth=.15)

ax.set_axis_off(); ax.set_xlim(-103.57, -103.14); ax.set_ylim(20.44, 20.82)
ax.legend(handles=[Patch(facecolor=c, label=e) for c, e in zip(RAMPA_SEQ, ETIQ)],
          title="Robos por 100 mil hab.", loc='upper left', fontsize=9,
          title_fontsize=9.5, labelspacing=.55, handlelength=1.4, handleheight=1.2, borderpad=.9)

for nombre, xy in [("GUADALAJARA", (-103.335, 20.700)), ("ZAPOPAN", (-103.452, 20.757)),
                   ("TLAQUEPAQUE", (-103.300, 20.588)), ("TONALÁ", (-103.218, 20.645)),
                   ("TLAJOMULCO", (-103.420, 20.490)), ("EL SALTO", (-103.180, 20.517))]:
    ax.annotate(nombre, xy, fontsize=8.5, color=INK, ha='center', fontweight='bold',
                path_effects=[pe.withStroke(linewidth=3, foreground=SURF)])

ax.text(0, 1.085, "El robo se concentra en el núcleo central, no en la periferia",
        transform=ax.transAxes, fontsize=14, fontweight='bold', color=INK, va='bottom')
ax.text(0, 1.028,
        f"Tasa de robo por colonia · Área Metropolitana de Guadalajara · jul-2025 a jul-2026 · "
        f"{len(mapa):,} colonias de 1 000 o más habitantes\n"
        "Fuentes: IIEG Jalisco (robos) y CONAPO, cartografía y población de colonias 2020",
        transform=ax.transAxes, fontsize=9.5, color=INK2, va='bottom')
nota(fig, "Se excluyen las colonias de menos de 1 000 habitantes porque su denominador pequeño dispara la tasa.\n"
          "Cortes fijos y no cuantiles, para que las clases sean comparables con otros mapas.")
fig.savefig(SALIDA/"fig5_h3_mapa.png"); plt.show()

print("Colonias con mayor tasa (1 000+ habitantes):\n")
print(mapa.nlargest(10, 'tasa')[['COLONIA','NOM_MUN','GM_2020','POBTOT','robos','tasa']]
          .round(0).to_string(index=False))

**Lectura.** El mapa muestra que la concentración no es periférica sino **central**: la mancha
oscura cubre el centro y poniente de Guadalajara y se extiende hacia el norte de Tlaquepaque,
mientras que la periferia de Tlajomulco, Tonalá y El Salto aparece en tonos claros.

Las colonias con mayor tasa combinan dos perfiles distintos, y conviene no confundirlos:

- **Corredor de ingreso alto**: Americana, Arcos Vallarta, Italia Providencia, Chapalita Oriente
  —justamente el patrón que predecía H3.
- **Zonas de alta actividad comercial y poca población residente**: Prados del Nilo y Obrera, por
  ejemplo, donde la tasa se dispara porque el denominador —habitantes— es pequeño frente al flujo
  diario de vehículos que llegan de fuera. En rigor, la tasa por residentes sobreestima el riesgo
  en estas colonias.

---
# Conclusiones

In [ ]:
resumen = pd.DataFrame({
 'Hipótesis': ['H1 · Patrón horario', 'H2 · Modus operandi', 'H3 · Factor socioeconómico'],
 'Enunciado inicial': ['Mayoría de robos de noche o madrugada',
                       'Predomina el robo con violencia',
                       'Mayor frecuencia en zonas de altos ingresos'],
 'Veredicto': ['SE REFUTA', 'SE REFUTA', 'SE CONFIRMA'],
 'Evidencia': [f"Distribución bimodal; madrugada 00–05 sólo {pct.loc[0:5].sum():.1f} %; diurno {pct.loc[7:18].sum():.1f} %",
               "Sin violencia es mayoría los 11 años; máximo violento 42.9 % (2020)",
               f"{tabla_h3.loc['Muy bajo','tasa_100k']:.0f} vs {tabla_h3.loc['Alto','tasa_100k']:.0f} robos por 100 mil hab."],
 'Fuente': ['IIEG Jalisco', 'SESNSP', 'IIEG + CONAPO'],
})
pd.set_option('display.max_colwidth', None)
print(resumen.to_string(index=False))
resumen.to_csv(SALIDA/"resumen_hipotesis.csv", index=False, encoding='utf-8-sig')
print("\nFiguras generadas en la carpeta:", SALIDA.resolve())
for f in sorted(SALIDA.glob("*.png")): print("  -", f.name)

### Síntesis

De las tres hipótesis iniciales, **dos se refutan y una se confirma**.

El robo de vehículos en Jalisco **no** es un delito nocturno ni mayoritariamente violento. Es un
delito que sigue el pulso de la ciudad: ocurre cuando la gente se mueve —picos a las 7 de la mañana
y entre las 8 y las 10 de la noche—, casi siempre sin confrontación directa con la víctima, y se
concentra en el núcleo urbano central antes que en la periferia.

La única creencia que resistió el contraste con los datos fue la socioeconómica, y aun ésa con un
matiz importante: el gradiente existe, pero su mecanismo probable es la **densidad de vehículos**
—hay más autos donde hay más ingreso— y no el valor comercial de cada unidad, que era la
explicación propuesta en el planteamiento original.

### Qué habría que hacer después

- **Confirmar con el IIEG** si el campo `fecha` registra el hecho o la denuncia. De ello depende
  que la caída de fin de semana sea un hallazgo o un artefacto.
- **Conseguir el parque vehicular por colonia o AGEB** para cambiar el denominador de H3 de
  habitantes a vehículos, y así separar el efecto «hay más autos» del efecto «los autos valen más».
- **Contrastar H1 y H2 con la ENVIPE**, que capta los delitos no denunciados y permitiría estimar
  cuánto del patrón observado se debe al sesgo de denuncia.

### Referencias

- CONAPO (2021). *Índices de marginación 2020. Índice de marginación urbana por colonia.*
  Consejo Nacional de Población. https://www.gob.mx/conapo/documentos/indices-de-marginacion-2020-284372
- IIEG Jalisco (2026). *Base georreferenciada de incidencia delictiva.* Instituto de Información
  Estadística y Geográfica de Jalisco.
- IIEG Jalisco (2019). *Robo de vehículos en Jalisco.* Documento de análisis, diciembre de 2019.
- SESNSP (2026). *Incidencia delictiva del fuero común, nueva metodología, 2015–2025.* Secretariado
  Ejecutivo del Sistema Nacional de Seguridad Pública.
- INEGI (2023). *Encuesta Nacional de Victimización y Percepción sobre Seguridad Pública (ENVIPE).*
- Knaflic, C. N. (2015). *Storytelling with Data: A Data Visualization Guide for Business
  Professionals.* Wiley.
- Bruce, P., Bruce, A. y Gedeck, P. (2020). *Practical Statistics for Data Scientists* (2.ª ed.).
  O'Reilly.

### Nota sobre el código

Todo el código de este notebook es original y fue desarrollado para este proyecto. Las únicas
dependencias externas son bibliotecas de uso estándar (`pandas`, `numpy`, `geopandas`,
`matplotlib`). La paleta de color sigue criterios de accesibilidad verificados con un validador
de separación perceptual en espacio OKLab bajo simulación de daltonismo.